In [0]:
# =============================================================================
# Notebook  : 02b_map_08_email_events_mapped
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_08_email_events_mapped
# Spec Ref  : §1.6 Event Aggregation — Email-Level (email_events_mapped)
# Purpose   : Aggregate mapped_events per email × meta_event × event_day.
#             email_events_mapped: email, meta_event, event_day, occurrences
#
# INCREMENTAL MERGE strategy:
#   Reads last aggregated event_day from target table.
#   Only re-aggregates events since that date.
#   For 1B BQ events this prevents rebuilding the full history every 15 min.
#
# Spec DQ gate:
#   email_events_mapped should cover all events with resolvable email
#   Compare COUNT(DISTINCT email) between mapped_events and email_events_mapped
#
# Run after : map_04 (mapped_events)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/FINAL_pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 08: email_events_mapped (incremental MERGE) ===  customer={customer_id}")

In [0]:
# CELL 3 -- Create table if not exists (idempotent)
create_map_table(
    f"{sv}.email_events_mapped",
    """
        email        STRING NOT NULL,
        meta_event   STRING NOT NULL,
        event_day    DATE   NOT NULL,
        occurrences  BIGINT,
        tenant       BIGINT
    """,
    cluster_by="email, meta_event, event_day"
)


In [0]:
# CELL 4 -- Find last aggregated day (watermark for incremental processing)
last_day = spark.sql(f"""
    SELECT COALESCE(MAX(event_day), DATE('1900-01-01'))
    FROM {sv}.email_events_mapped
""").collect()[0][0]
print(f"  Last aggregated day: {last_day}  (reprocessing from this date)")

# CELL 5 -- Aggregate only events since last_day
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW email_events_delta AS
    SELECT
        c.email,
        e.meta_event,
        CAST(e.event_timestamp AS DATE)   AS event_day,
        COUNT(*)                          AS occurrences,
        e.tenant
    FROM {sv}.mapped_events e
    INNER JOIN {sv}.contacts_all c
        ON e.contactid = c.id
    WHERE c.email IS NOT NULL
      AND CAST(e.event_timestamp AS DATE) >= '{last_day}'
    GROUP BY
        c.email,
        e.meta_event,
        CAST(e.event_timestamp AS DATE),
        e.tenant
""")

# Add tenant column if missing (table may pre-date DDL update)
existing_cols = [c.name for c in spark.table(f"{sv}.email_events_mapped").schema]
if "tenant" not in existing_cols:
    spark.sql(f"ALTER TABLE {sv}.email_events_mapped ADD COLUMN tenant BIGINT")

# CELL 6 -- MERGE: update existing + insert new
spark.sql(f"""
    MERGE INTO {sv}.email_events_mapped AS target
    USING email_events_delta AS source
    ON  target.email      = source.email
    AND target.meta_event = source.meta_event
    AND target.event_day  = source.event_day
    AND target.tenant     = source.tenant
    WHEN MATCHED THEN
        UPDATE SET
            target.occurrences = source.occurrences
    WHEN NOT MATCHED THEN
        INSERT (email, meta_event, event_day, occurrences, tenant)
        VALUES (source.email, source.meta_event, source.event_day,
                source.occurrences, source.tenant)
""")


# CELL 7 -- Spec DQ check
n = cnt(f"{sv}.email_events_mapped")
bad_domain = spark.sql(f"""
    SELECT COUNT(*) FROM {sv}.email_events_mapped
    WHERE email IS NULL OR TRIM(email) = ''
""").collect()[0][0]
print(f"  email_events_mapped: {n:,} rows")
print(f"  Null/empty email rows: {bad_domain}  (should be 0)")
dbutils.notebook.exit("Success")